In [8]:
# ============================================================
# Cell 1:
# 对每个模型做 PCA(n=50)，并计算 PC1-PC50 与 eye_DR_Level 的 Spearman rank
# 输出:
# - pca_results: 每个模型的 PCA 后 DataFrame
# - pca_model_info_df: 每个模型的 PCA 基本信息
# - dr_spearman_all_df: 所有模型、所有PC与 DR 的 Spearman 结果
# - model_dr_summary_df: 每个模型 DR 最强相关PC汇总
# ============================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# ------------------------------------------------------------
# 0. 基本设置
# ------------------------------------------------------------

BASE_DIR = Path("./aligned_features_drvalid")
N_PC = 50
DR_COL = "eye_DR_Level"
RANDOM_STATE = 0
# 如果只想分析指定 VLM 模型，在这里写模型名。
# 如果设为 None，则自动读取 BASE_DIR 下所有 aligned_drvalid_*.npz。
# 例:
# MODEL_NAMES = [
#     "FVs_medsiglip",
#     "FVs_FLAIR",
#     "FVs_biomed",
#     "FVs_ConceptCLIP",
#     "FVs_CXRCLIP",
#     "FVs_CONCH",
#     "FVs_llava-med",
#     "FVs_LLM2CLIP-EVA",
#     "FVs_MONET",
#     "FVs_siglip",
# ]
MODEL_NAMES = None

# ------------------------------------------------------------
# 1. 工具函数
# ------------------------------------------------------------

def discover_model_names(base_dir=BASE_DIR):
    """从 aligned_drvalid_*.npz 自动发现模型名。"""
    npz_files = sorted(base_dir.glob("aligned_drvalid_*.npz"))
    model_names = []
    for p in npz_files:
        name = p.stem.replace("aligned_drvalid_", "")
        # 安全起见，排除异常命名
        if name and not name.endswith("_metadata"):
            model_names.append(name)
    return model_names


def load_aligned_feature(model_name, base_dir=BASE_DIR):
    """读取某个模型的 aligned feature 和 metadata。"""
    npz_path = base_dir / f"aligned_drvalid_{model_name}.npz"
    meta_path = base_dir / f"aligned_drvalid_{model_name}_metadata.csv"

    if not npz_path.exists():
        raise FileNotFoundError(f"Feature file not found: {npz_path}")

    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {meta_path}")

    data = np.load(npz_path, allow_pickle=True)
    X = data["X"]
    meta_df = pd.read_csv(meta_path)

    if len(meta_df) != X.shape[0]:
        raise ValueError(
            f"{model_name}: X rows != metadata rows: {X.shape[0]} vs {len(meta_df)}"
        )

    return X, meta_df


def run_pca_for_model(
    X,
    meta_df,
    model_name,
    n_pc=N_PC,
    random_state=RANDOM_STATE
):
    """标准化后做 PCA，并返回 pca_df, pca对象, scaler对象。"""
    X_raw = np.asarray(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    n_components = min(n_pc, X_scaled.shape[0], X_scaled.shape[1])

    pca = PCA(n_components=n_components, random_state=random_state)
    Z = pca.fit_transform(X_scaled)

    pca_df = meta_df.copy()
    pca_df["model_name"] = model_name

    for i in range(Z.shape[1]):
        pca_df[f"PC{i+1}"] = Z[:, i]

    return pca_df, pca, scaler


def compute_pc_spearman_rank(
    df,
    label_col=DR_COL,
    n_pc=N_PC,
    pc_prefix="PC",
    model_name=None
):
    """PC1-PCn Spearman rank """
    rows = []

    for i in range(1, n_pc + 1):
        pc = f"{pc_prefix}{i}"

        if pc not in df.columns or label_col not in df.columns:
            continue

        tmp = df[[pc, label_col]].copy()
        tmp[pc] = pd.to_numeric(tmp[pc], errors="coerce")
        tmp[label_col] = pd.to_numeric(tmp[label_col], errors="coerce")
        tmp = tmp.dropna()

        if len(tmp) < 3 or tmp[pc].nunique() < 2 or tmp[label_col].nunique() < 2:
            rho, p_value = np.nan, np.nan
        else:
            rho, p_value = spearmanr(tmp[pc], tmp[label_col])

        rows.append({
            "model_name": model_name,
            "PC": pc,
            "factor": label_col,
            "n": len(tmp),
            "spearman_rho": rho,
            "abs_spearman_rho": abs(rho) if pd.notna(rho) else np.nan,
            "p_value": p_value,
        })

    result_df = pd.DataFrame(rows)
    if len(result_df) > 0:
        result_df = result_df.sort_values("abs_spearman_rho", ascending=False).reset_index(drop=True)
        result_df["rank"] = np.arange(1, len(result_df) + 1)

    return result_df

# ------------------------------------------------------------
# 2. 自动确定模型列表
# ------------------------------------------------------------

if MODEL_NAMES is None:
    MODEL_NAMES = discover_model_names(BASE_DIR)

print("Models to analyze:")
for m in MODEL_NAMES:
    print(" -", m)

# ------------------------------------------------------------
# 3. 对每个模型做 PCA + DR Spearman rank
# ------------------------------------------------------------

pca_results = {}
pca_objects = {}
scaler_objects = {}

pca_info_rows = []
dr_rank_dfs = []
model_summary_rows = []

for model_name in MODEL_NAMES:
    print("\n" + "=" * 80)
    print(f"Processing model: {model_name}")

    try:
        X, meta_df = load_aligned_feature(model_name, BASE_DIR)
        pca_df, pca, scaler = run_pca_for_model(
            X=X,
            meta_df=meta_df,
            model_name=model_name,
            n_pc=N_PC,
            random_state=RANDOM_STATE
        )

        pca_results[model_name] = pca_df
        pca_objects[model_name] = pca
        scaler_objects[model_name] = scaler

        pca_info_rows.append({
            "model_name": model_name,
            "n_samples": X.shape[0],
            "feature_dim": X.shape[1],
            "n_pc": pca.n_components_,
            "explained_var_PC1": pca.explained_variance_ratio_[0],
            "explained_var_top10": pca.explained_variance_ratio_[:10].sum(),
            "explained_var_top50": pca.explained_variance_ratio_.sum(),
        })

        dr_rank_df = compute_pc_spearman_rank(
            df=pca_df,
            label_col=DR_COL,
            n_pc=pca.n_components_,
            pc_prefix="PC",
            model_name=model_name
        )

        dr_rank_dfs.append(dr_rank_df)

        top_row = dr_rank_df.iloc[0].to_dict()

        model_summary_rows.append({
            "model_name": model_name,
            "top_DR_PC": top_row["PC"],
            "top_DR_spearman_rho": top_row["spearman_rho"],
            "top_DR_abs_spearman_rho": top_row["abs_spearman_rho"],
            "top_DR_p_value": top_row["p_value"],
            "n_valid": top_row["n"],
            "n_pc": pca.n_components_,
            "explained_var_top50": pca.explained_variance_ratio_.sum(),
        })

        # print(f"X shape: {X.shape}")
        # print(f"PCA shape: {pca_df[[f'PC{i}' for i in range(1, pca.n_components_ + 1)]].shape}")
        # print("Top DR-related PCs:")
        # display(dr_rank_df.head(10))

    except Exception as e:
        print(f"[ERROR] {model_name}: {e}")

# ------------------------------------------------------------
# 4. 汇总表
# ------------------------------------------------------------

pca_model_info_df = pd.DataFrame(pca_info_rows)
dr_spearman_all_df = pd.concat(dr_rank_dfs, ignore_index=True) if dr_rank_dfs else pd.DataFrame()
model_dr_summary_df = pd.DataFrame(model_summary_rows)

print("\nPCA model info:")
display(pca_model_info_df)

print("\nModel-level DR strongest PC summary:")
display(
    model_dr_summary_df
    .sort_values("top_DR_abs_spearman_rho", ascending=False)
    .reset_index(drop=True)
)

# OUT_DIR = Path("./pca_all_models_results")
# OUT_DIR.mkdir(exist_ok=True)

# pca_model_info_df.to_csv(OUT_DIR / "pca_model_info.csv", index=False)
# dr_spearman_all_df.to_csv(OUT_DIR / "dr_spearman_all_models.csv", index=False)
# model_dr_summary_df.to_csv(OUT_DIR / "model_dr_summary.csv", index=False)

# print(f"\nSaved summary CSVs to: {OUT_DIR.resolve()}")


Models to analyze:
 - FVs_biomed
 - FVs_ConceptCLIP
 - FVs_CONCH
 - FVs_CXRCLIP
 - FVs_DINOv2
 - FVs_EVA02-L-14-336
 - FVs_FLAIR
 - FVs_INViT-L-16
 - FVs_llava-med
 - FVs_llava-Mistral-7b
 - FVs_llava-vicuna-13b-hf
 - FVs_LLM2CLIP-EVA
 - FVs_LLM2CLIP-openai
 - FVs_medsiglip
 - FVs_MedTrinity
 - FVs_MONET
 - FVs_resnet50
 - FVs_SigLIP
 - FVs_swin
 - FVs_UNI
 - FVs_VGG16
 - FVs_ViT-B-16
 - FVs_ViT-g-14
 - FVs_ViT-L-14

Processing model: FVs_biomed

Processing model: FVs_ConceptCLIP

Processing model: FVs_CONCH

Processing model: FVs_CXRCLIP

Processing model: FVs_DINOv2

Processing model: FVs_EVA02-L-14-336

Processing model: FVs_FLAIR

Processing model: FVs_INViT-L-16

Processing model: FVs_llava-med

Processing model: FVs_llava-Mistral-7b

Processing model: FVs_llava-vicuna-13b-hf

Processing model: FVs_LLM2CLIP-EVA

Processing model: FVs_LLM2CLIP-openai

Processing model: FVs_medsiglip

Processing model: FVs_MedTrinity

Processing model: FVs_MONET

Processing model: FVs_resnet50

Proc

,model_name,n_samples,feature_dim,n_pc,explained_var_PC1,explained_var_top10,explained_var_top50
0,FVs_biomed,1189,512,50,0.148771,0.632739,0.926510
1,FVs_ConceptCLIP,1189,1152,50,0.123416,0.656470,0.904868
2,FVs_CONCH,1189,512,50,0.157774,0.666059,0.959024
3,FVs_CXRCLIP,1189,512,50,0.627539,0.869708,0.970453
4,FVs_DINOv2,1189,1024,50,0.132956,0.579693,0.845378
5,FVs_EVA02-L-14-336,1189,768,50,0.175580,0.646163,0.918347
6,FVs_FLAIR,1189,512,50,0.232017,0.865229,0.999317
7,FVs_INViT-L-16,1189,1024,50,0.185883,0.592471,0.831346
8,FVs_llava-med,1189,4096,50,0.134901,0.544153,0.816057
9,FVs_llava-Mistral-7b,1189,4096,50,0.145790,0.584432,0.867189



Model-level DR strongest PC summary:


,model_name,top_DR_PC,top_DR_spearman_rho,top_DR_abs_spearman_rho,top_DR_p_value,n_valid,n_pc,explained_var_top50
0,FVs_FLAIR,PC1,0.788109,0.788109,2.046253e-252,1189,50,0.999317
1,FVs_medsiglip,PC1,-0.787509,0.787509,8.999141e-252,1189,50,0.877816
2,FVs_LLM2CLIP-openai,PC1,0.666696,0.666696,1.042217e-153,1189,50,0.866201
3,FVs_ConceptCLIP,PC1,0.642148,0.642148,3.361219e-139,1189,50,0.904868
4,FVs_MONET,PC2,-0.608878,0.608878,1.543484e-121,1189,50,0.851570
5,FVs_SigLIP,PC1,-0.594220,0.594220,2.103421e-114,1189,50,0.873693
6,FVs_UNI,PC1,-0.573277,0.573277,7.965340e-105,1189,50,0.857237
7,FVs_ViT-g-14,PC2,-0.557271,0.557271,5.869754e-98,1189,50,0.828402
8,FVs_EVA02-L-14-336,PC1,-0.548904,0.548904,1.631294e-94,1189,50,0.918347
9,FVs_resnet50,PC2,-0.530041,0.530041,4.246809e-87,1189,50,0.791414


#### selected_models (top5 abs_rho>0.6)
- FLAIR
- medsiglip
- LLM2CLIP-openai
- ConceptCLIP
- MONET
- SigLIP (abs_rho = 0.594 compare with medsiglip)
